In [1]:
!pip install streamlit pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 5.2 MB/s eta 0:00:00


In [2]:
!ngrok authtoken 2uwi3Qa6ImQbQWGdMapfseN4Axj_fyTRGXvqgMz4SVzQxBrT

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [29]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# Load your model and encoders
le_team = joblib.load('/content/le_team123.pkl')
le_venue = joblib.load('/content/le_venue123.pkl')
model = joblib.load('/content/final_rf_model.pkl')

# Streamlit Page Config
st.set_page_config(layout="wide", page_title="🏏 Cricket Match Predictor", page_icon="🏆")

# Title with Emoji
st.markdown("<h1 style='text-align: center; color: #ff5733;'>🏏 Cricket Match Predictor</h1>", unsafe_allow_html=True)

# Sidebar for Input Features
st.sidebar.header("🔢 Match Input Features")
inning = st.sidebar.number_input("📌 Inning", min_value=1, max_value=2, value=1)
cum_runs = st.sidebar.number_input("🏏 Cumulative Runs", min_value=0, value=80)
cum_wickets = st.sidebar.number_input("❌ Cumulative Wickets", min_value=0, value=5)
overs_completed = st.sidebar.number_input("⏳ Overs Completed", min_value=0.0, value=12.0)
target = st.sidebar.number_input("🎯 Target Score", min_value=0, value=0)
batting_team = st.sidebar.selectbox(" 🏏Batting Team", le_team.classes_)
bowling_team = st.sidebar.selectbox(" Bowling Team", le_team.classes_)
venue = st.sidebar.selectbox("📍 Venue", le_venue.classes_)

# Prediction Button
if st.sidebar.button("🚀 Predict Match Outcome"):
    # Feature Engineering
    current_run_rate = cum_runs / max(overs_completed, 1)
    remaining_overs = max(20 - overs_completed, 1)
    required_run_rate = (target - cum_runs) / remaining_overs

    batting_team_encoded = le_team.transform([batting_team])[0]
    bowling_team_encoded = le_team.transform([bowling_team])[0]
    venue_encoded = le_venue.transform([venue])[0]

    input_df = pd.DataFrame([{
        "inning": inning,
        "cum_runs": cum_runs,
        "cum_wickets": cum_wickets,
        "overs_completed": overs_completed,
        "target": target,
        "batting_team": batting_team_encoded,
        "bowling_team": bowling_team_encoded,
        "venue_encoded": venue_encoded,
        "current_run_rate": current_run_rate,
        "required_run_rate": required_run_rate
    }])

    input_df = input_df[model.feature_names_in_]
    predicted_probabilities = model.predict_proba(input_df)[0]

    # Determine Winner and Loser
    teams = [batting_team, bowling_team]
    winner_index = np.argmax(predicted_probabilities)
    predicted_winner = teams[winner_index]
    predicted_loser = teams[1 - winner_index]

    # Colors for Teams
    colors = ["#00A86B" if team == predicted_winner else "#D72638" for team in teams]

    # Result Display
    st.markdown(f"""
        <div style="text-align: center; font-size: 48px; font-weight: bold; color: #00A86B;">
            🎉 Predicted Winner: <span style="color: #FFD700;">{predicted_winner}</span> 🏆
        </div>
    """, unsafe_allow_html=True)

    # Progress Bar for Probability
    st.markdown(f"### 🏆 Win Probability")
    st.progress(float(predicted_probabilities[winner_index]))

    # Enhanced Probability Chart
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.barplot(x=teams, y=predicted_probabilities, palette=colors, ax=ax)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Winning Probability", fontsize=12)
    ax.set_xlabel("Teams", fontsize=12)
    ax.set_title("📊 Match Win Probability", fontsize=14)

    # Show Probability on Bars
    for i, v in enumerate(predicted_probabilities):
        ax.text(i, v + 0.02, f"{v:.2%}", ha='center', fontsize=12, fontweight='bold')

    # Show Chart in Streamlit
    st.pyplot(fig)




Overwriting app.py


In [10]:
import time
from pyngrok import ngrok

# ✅ Ensure ngrok is properly authenticated
! ngrok config add-authtoken  2uwi3Qa6ImQbQWGdMapfseN4Axj_fyTRGXvqgMz4SVzQxBrT# Replace with your ngrok token

# ✅ Start Streamlit in the background
!nohup streamlit run app.py --server.port 8501 &

# ✅ Wait for Streamlit to start
time.sleep(5)

# ✅ Manually configure ngrok with the correct settings
public_url = ngrok.connect("http://localhost:8501")
print(f"📢 Open this URL to access your Streamlit app: {public_url}")

ERROR:  accepts 1 arg(s), received 6
nohup: appending output to 'nohup.out'


PyngrokNgrokHTTPError: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: Your account may not run more than 3 tunnels over a single ngrok agent session.\nThe tunnels already running on this session are:\ntn_2uwiKjdaRawZJKg0b4ShgrVc8gE, tn_2uwiXtTYD2C1U6JfXhPYDThrsBg, tn_2uwjGwflGjD1CnGKrJ6d5nD8rw4\n\r\n\r\nERR_NGROK_324\r\n"}}
